# Week 24, Day 6: Forge

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w024d6.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
import pytest
print(0.1 * 3 == 0.3, 0.1 * 3 == pytest.approx(0.3))

**Predict before you run:** what will this print?



In [ ]:
def test_nothing():
    1 + 1 == 3

test_nothing()
print("passed?")

**Predict before you run:** what will this print?



In [ ]:
import pytest

def check(n):
    if n < 0:
        raise ValueError("negative")
    return n

try:
    with pytest.raises(ValueError):
        check(5)
except BaseException as e:
    print(type(e).__name__)

**Predict before you run:** what will this print?



In [ ]:
import random
a = [random.Random(4).randint(1, 6) for _ in range(3)]
rng = random.Random(4)
b = [rng.randint(1, 6) for _ in range(3)]
print(a == [a[0]] * 3, len(set(b)) >= 1)

**Predict before you run:** what will this print?



In [ ]:
from datetime import date, timedelta
d = date(2026, 1, 31) + timedelta(days=29)
print(d, d.weekday())

### Exercise: Parametrize the cases

`slug(title)` in `slugs.py` turns a title into a web address part: lower case, words joined by `-`, and anything that isn't a letter, digit or space removed (`"Hello, World!"` becomes `"hello-world"`). Write one parametrized test with at least five cases that catches every broken version.

In [ ]:
# Run this cell first: it creates the files used below.
with open('slugs.py', 'w', encoding='utf-8') as f:
    f.write('import re\n\ndef slug(title):\n    cleaned = re.sub(r"[^a-z0-9 ]", "", title.lower())\n    return "-".join(cleaned.split())\n')

In [ ]:
import pytest
from slugs import slug

In [ ]:
# Check: run the exercise cell above first, then this one.
check("GOOD = open(\"slugs.py\").read()\nBROKEN = {\n    \"keeps capitals\": GOOD.replace(\"title.lower()\", \"title\"),\n    \"keeps punctuation\": GOOD.replace('re.sub(r\"[^a-z0-9 ]\", \"\", title.lower())', \"title.lower()\"),\n    \"single spaces only\": GOOD.replace(\"cleaned.split()\", 'cleaned.split(\" \")'),\n    \"drops digits\": GOOD.replace(\"a-z0-9\", \"a-z\"),\n}\ndef test_style():\n    \"one parametrized test with at least five cases\"\n    assert \"parametrize\" in source\n    assert len(pytest_run(source)[\"passed\"]) >= 5\ndef test_catches():\n    \"it catches every broken version\"\n    results = against(source, \"slugs\", {\"correct\": GOOD, **BROKEN})\n    write_file(\"slugs.py\", GOOD)\n    assert results[\"correct\"] == [], f\"these fail on the correct code: {results['correct']}\"\n    missed = [k for k in BROKEN if not results[k]]\n    assert not missed, f\"not caught: {missed}\"")

### Exercise: A property of reversing

`reverse_words(text)` in `words.py` reverses the order of the words in a text (words are separated by single spaces). Write a seeded property test: for many random lists of words, joining them with spaces, reversing twice gives the text back, and reversing once gives the words in reverse order. Add one example test too.

In [ ]:
# Run this cell first: it creates the files used below.
with open('words.py', 'w', encoding='utf-8') as f:
    f.write('def reverse_words(text):\n    return " ".join(reversed(text.split(" ")))\n')

In [ ]:
import random
from words import reverse_words

In [ ]:
# Check: run the exercise cell above first, then this one.
check("GOOD = open(\"words.py\").read()\nBROKEN = {\n    \"does nothing\": \"def reverse_words(text):\\n    return text\\n\",\n    \"reverses letters\": \"def reverse_words(text):\\n    return text[::-1]\\n\",\n    \"drops one word\": \"def reverse_words(text):\\n    return ' '.join(list(reversed(text.split(' ')))[:3])\\n\",\n}\ndef test_style():\n    \"a seeded property test\"\n    assert \"Random(\" in source\ndef test_catches():\n    \"your tests pass on the correct code and catch every broken version\"\n    results = against(source, \"words\", {\"correct\": GOOD, **BROKEN})\n    write_file(\"words.py\", GOOD)\n    assert results[\"correct\"] == [], f\"these fail on the correct code: {results['correct']}\"\n    missed = [k for k in BROKEN if not results[k]]\n    assert not missed, f\"not caught: {missed}\"")

### Exercise: Green, then refactor

Make the tests in `test_temps.py` pass by writing `classify(celsius)` in `temps.py`. Use no `elif`: loop over a list of `(limit, label)` pairs instead.

In [ ]:
# Run this cell first: it creates the files used below.
with open('test_temps.py', 'w', encoding='utf-8') as f:
    f.write('import pytest\nfrom temps import classify\n\n@pytest.mark.parametrize("c, label", [\n    (-5, "freezing"), (0, "freezing"), (0.1, "cold"), (10, "cold"),\n    (10.5, "mild"), (20, "mild"), (21, "warm"), (30, "warm"), (30.5, "hot"),\n])\ndef test_classify(c, label):\n    assert classify(c) == label\n')

In [ ]:
BANDS = [(0, "freezing"), (10, "cold"), (20, "mild"), (30, "warm")]

def classify(celsius):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("TESTS = open(\"test_temps.py\").read()\ndef test_green():\n    \"test_temps.py passes\"\n    r = pytest_run(TESTS, files={\"temps.py\": source})\n    assert not r[\"failed\"] and len(r[\"passed\"]) == 9, f\"failing: {r['failed']}\"\ndef test_no_elif():\n    \"no elif\"\n    assert \"elif\" not in source")

### Exercise: Captured output

`countdown(n)` in `launch.py` prints `n` down to 1, one per line, then `Lift off!`; for `n` of 0 it prints only `Lift off!`; a negative `n` raises `ValueError`. Write tests with `capsys` and `pytest.raises` that catch every broken version.

In [ ]:
# Run this cell first: it creates the files used below.
with open('launch.py', 'w', encoding='utf-8') as f:
    f.write('def countdown(n):\n    if n < 0:\n        raise ValueError("n can\'t be negative")\n    for i in range(n, 0, -1):\n        print(i)\n    print("Lift off!")\n')

In [ ]:
import pytest
from launch import countdown

In [ ]:
# Check: run the exercise cell above first, then this one.
check("GOOD = open(\"launch.py\").read()\nBROKEN = {\n    \"includes 0\": GOOD.replace(\"range(n, 0, -1)\", \"range(n, -1, -1)\"),\n    \"counts up\": GOOD.replace(\"range(n, 0, -1)\", \"range(1, n + 1)\"),\n    \"no lift off\": GOOD.replace('    print(\"Lift off!\")\\n', \"\"),\n    \"negative allowed\": GOOD.replace('    if n < 0:\\n        raise ValueError(\"n can\\'t be negative\")\\n', \"\"),\n}\ndef test_catches():\n    \"your tests pass on the correct code and catch every broken version\"\n    assert \"capsys\" in source\n    results = against(source, \"launch\", {\"correct\": GOOD, **BROKEN})\n    write_file(\"launch.py\", GOOD)\n    assert results[\"correct\"] == [], f\"these fail on the correct code: {results['correct']}\"\n    missed = [k for k in BROKEN if not results[k]]\n    assert not missed, f\"not caught: {missed}\"")

### Exercise: Forge: a late-fee calculator, test first

Write `late_fee(due, returned, kind, age)` returning the fee in whole pence, where `due` and `returned` are `date`s, `kind` is `"book"`, `"dvd"` or `"reference"`, and `age` is a whole number. Raise `ValueError` for a reference item or an unknown kind.

Work test first, in one editor: tests for each rule above, then the code. The checks:

1. run their own hidden tests on your `late_fee`;
2. run **your** tests on your `late_fee`, where they must all pass;
3. run your tests on **nine** broken versions, where they must catch every one.

Your tests need at least six `test_` functions (parametrized ones count once).

In [ ]:
import pytest
from datetime import date

def test_example_from_the_librarian():
    assert late_fee(date(2026, 11, 6), date(2026, 11, 10), "book", 30) == 45

def late_fee(due, returned, kind, age):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import ast as _ast\nfrom datetime import date as _d\nGOOD = '''\nfrom datetime import timedelta\n\nRATES = {\"book\": (15, 500), \"dvd\": (100, 1000)}\n\ndef days_late(due, returned):\n    days = 0\n    day = due + timedelta(days=1)\n    while day <= returned:\n        if day.weekday() != 6:\n            days += 1\n        day += timedelta(days=1)\n    return days\n\ndef late_fee(due, returned, kind, age):\n    if kind not in RATES:\n        raise ValueError(f\"can't charge for {kind!r}\")\n    rate, cap = RATES[kind]\n    fee = min(days_late(due, returned) * rate, cap)\n    if age < 16:\n        fee //= 2\n    return fee\n'''\nBROKEN = {\n    \"counts Sundays\": GOOD.replace(\"if day.weekday() != 6:\", \"if True:\"),\n    \"counts Saturdays instead\": GOOD.replace(\"weekday() != 6\", \"weekday() != 5\"),\n    \"counts the due day\": GOOD.replace(\"day = due + timedelta(days=1)\", \"day = due\"),\n    \"misses the return day\": GOOD.replace(\"while day <= returned:\", \"while day < returned:\"),\n    \"no book cap\": GOOD.replace('\"book\": (15, 500)', '\"book\": (15, 10**9)'),\n    \"dvd rate wrong\": GOOD.replace('\"dvd\": (100, 1000)', '\"dvd\": (50, 1000)'),\n    \"16 is half price\": GOOD.replace(\"age < 16\", \"age <= 16\"),\n    \"half rounds up\": GOOD.replace(\"fee //= 2\", \"fee = -(-fee // 2)\"),\n    \"reference is free\": GOOD.replace(\"if kind not in RATES:\", \"if kind == 'reference':\\n        return 0\\n    if kind not in RATES:\"),\n}\ndef test_hidden():\n    \"your late_fee passes the hidden tests\"\n    cases = [\n        ((_d(2026, 11, 6), _d(2026, 11, 10), \"book\", 30), 45),\n        ((_d(2026, 11, 6), _d(2026, 11, 6), \"book\", 30), 0),\n        ((_d(2026, 11, 6), _d(2026, 11, 1), \"dvd\", 30), 0),\n        ((_d(2026, 11, 7), _d(2026, 11, 8), \"dvd\", 30), 0),\n        ((_d(2026, 11, 7), _d(2026, 11, 9), \"dvd\", 30), 100),\n        ((_d(2026, 11, 2), _d(2026, 12, 31), \"book\", 30), 500),\n        ((_d(2026, 11, 2), _d(2026, 12, 31), \"dvd\", 40), 1000),\n        ((_d(2026, 11, 2), _d(2026, 11, 5), \"book\", 15), 22),\n        ((_d(2026, 11, 2), _d(2026, 11, 3), \"book\", 10), 7),\n        ((_d(2026, 11, 2), _d(2026, 11, 3), \"dvd\", 16), 100),\n        ((_d(2026, 11, 2), _d(2026, 11, 4), \"book\", 12), 15),\n        ((_d(2026, 11, 2), _d(2026, 11, 3), \"book\", 15), 7),\n        ((_d(2026, 12, 30), _d(2027, 1, 2), \"book\", 30), 45),\n    ]\n    for args, want in cases:\n        got = late_fee(*args)\n        assert got == want, f\"late_fee{args} should be {want}, not {got}\"\n    for kind in (\"reference\", \"magazine\"):\n        try:\n            late_fee(_d(2026, 11, 2), _d(2026, 11, 3), kind, 30)\n            assert False, f\"late_fee(..., {kind!r}, 30) should raise ValueError\"\n        except ValueError:\n            pass\ndef _tests_only():\n    tree = _ast.parse(source)\n    keep = [n for n in tree.body if isinstance(n, (_ast.Import, _ast.ImportFrom))\n            or (isinstance(n, _ast.FunctionDef) and n.name.startswith(\"test_\"))]\n    return \"from fees import late_fee\\n\" + _ast.unparse(_ast.Module(body=keep, type_ignores=[]))\ndef test_yours_pass():\n    \"you wrote at least six tests, and they all pass on your code\"\n    tree = _ast.parse(source)\n    names = [n.name for n in tree.body if isinstance(n, _ast.FunctionDef) and n.name.startswith(\"test_\")]\n    assert len(names) >= 6, f\"write at least six test functions (you have {len(names)})\"\n    r = pytest_run(source)\n    assert not r[\"failed\"], f\"these of your tests fail on your own code: {r['failed']}\"\ndef test_yours_catch():\n    \"your tests catch all nine broken versions (and pass on a correct one)\"\n    results = against(_tests_only(), \"fees\", {\"correct\": GOOD, **BROKEN})\n    assert results[\"correct\"] == [], f\"these fail on a correct late_fee: {results['correct']}\"\n    missed = [k for k in BROKEN if not results[k]]\n    assert not missed, f\"not caught: {missed}\"")

> **The tests are the specification.** Your tests now say, in code, exactly what the librarian told you: the Sunday rule, the caps, the half price for under-16s. If the library changes a rule next year, whoever updates the code changes a test first, sees it fail, and then changes the code. And if they break a rule they weren't meant to touch, a test says so before any member is charged the wrong amount.

### Exercise: Forge bonus: testing a ledger

`Ledger(path)` in `ledger.py` keeps a club's money records in a CSV file. Read its docstrings for exactly what each method should do. Write a test suite, using a fixture with `tmp_path`, `pytest.raises` and `pytest.approx` where they fit, that passes on the treasurer's code and catches all **eight** broken versions.

In [ ]:
# Run this cell first: it creates the files used below.
with open('ledger.py', 'w', encoding='utf-8') as f:
    f.write('import csv\nfrom pathlib import Path\n\n\nclass Ledger:\n    """Money in and out, saved to a CSV file with columns date, who, amount.\n    Amounts are in pounds: positive for money in, negative for money out."""\n\n    def __init__(self, path):\n        """Open a ledger file. A missing file means an empty ledger."""\n        self.path = Path(path)\n        self.rows = []\n        if self.path.exists():\n            with open(self.path, newline="", encoding="utf-8") as f:\n                for row in csv.DictReader(f):\n                    self.rows.append({"date": row["date"], "who": row["who"], "amount": float(row["amount"])})\n\n    def record(self, date, who, amount):\n        """Add an entry and save. who is stripped and can\'t be empty; amount can\'t be 0.\n        Both mistakes raise ValueError and record nothing."""\n        who = who.strip()\n        if not who:\n            raise ValueError("who is missing")\n        if amount == 0:\n            raise ValueError("amount can\'t be zero")\n        self.rows.append({"date": date, "who": who, "amount": amount})\n        self._save()\n\n    def _save(self):\n        with open(self.path, "w", newline="", encoding="utf-8") as f:\n            w = csv.DictWriter(f, fieldnames=["date", "who", "amount"])\n            w.writeheader()\n            w.writerows(self.rows)\n\n    def balance(self):\n        """Money in minus money out, rounded to 2 decimal places."""\n        return round(sum(r["amount"] for r in self.rows), 2)\n\n    def paid_in_by(self, who):\n        """Total paid in (positive amounts only) by one person, ignoring case."""\n        return round(sum(r["amount"] for r in self.rows\n                         if r["amount"] > 0 and r["who"].lower() == who.lower()), 2)\n\n    def largest_expense(self):\n        """The row with the most negative amount, or None if there\'s no money out."""\n        out = [r for r in self.rows if r["amount"] < 0]\n        return min(out, key=lambda r: r["amount"]) if out else None\n')

In [ ]:
import pytest
from ledger import Ledger

@pytest.fixture
def ledger(tmp_path):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("GOOD = open(\"ledger.py\").read()\nBROKEN = {\n    \"not saved\": GOOD.replace(\"        self.rows.append({\\\"date\\\": date, \\\"who\\\": who, \\\"amount\\\": amount})\\n        self._save()\",\n                              \"        self.rows.append({\\\"date\\\": date, \\\"who\\\": who, \\\"amount\\\": amount})\"),\n    \"amounts load as text\": GOOD.replace('\"amount\": float(row[\"amount\"])', '\"amount\": row[\"amount\"]').replace(\n        'sum(r[\"amount\"] for r in self.rows)', 'sum(float(r[\"amount\"]) for r in self.rows)'),\n    \"who not stripped\": GOOD.replace(\"        who = who.strip()\\n        if not who:\", \"        if not who.strip():\"),\n    \"zero allowed\": GOOD.replace('        if amount == 0:\\n            raise ValueError(\"amount can\\'t be zero\")\\n', \"\"),\n    \"bad entry still recorded\": GOOD.replace('        who = who.strip()\\n        if not who:\\n            raise ValueError(\"who is missing\")\\n',\n        '        who = who.strip()\\n        if not who:\\n            self.rows.append({\"date\": date, \"who\": who, \"amount\": amount})\\n            raise ValueError(\"who is missing\")\\n'),\n    \"paid in counts money out\": GOOD.replace('if r[\"amount\"] > 0 and r[\"who\"]', 'if r[\"who\"]'),\n    \"paid in is case-sensitive\": GOOD.replace('r[\"who\"].lower() == who.lower()', 'r[\"who\"] == who'),\n    \"largest expense is the smallest\": GOOD.replace(\"min(out, key\", \"max(out, key\"),\n}\ndef test_style():\n    \"a tmp_path fixture, and pytest.raises\"\n    assert \"@pytest.fixture\" in source and \"tmp_path\" in source and \"pytest.raises\" in source\ndef test_catches():\n    \"your tests pass on the treasurer's code and catch all eight broken versions\"\n    results = against(source, \"ledger\", {\"correct\": GOOD, **BROKEN})\n    write_file(\"ledger.py\", GOOD)\n    assert results[\"correct\"] == [], f\"these fail on the treasurer's code: {results['correct']}\"\n    missed = [k for k in BROKEN if not results[k]]\n    assert not missed, f\"not caught: {missed}\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

## Warm-up

If any surprised you, revisit [why we test](w024d1.html), [pytest](w024d2.html), [designing tests](w024d3.html), [TDD](w024d4.html) or [fixtures](w024d5.html). (The third: `pytest.raises` fails the test when nothing is raised, with an error pytest calls `Failed`. The fourth: a new generator with the same seed starts the same sequence every time.)

## Mixed review

## Forge: a late-fee calculator, test first

A community library wants its late fees worked out by a program instead of by hand. The rules, from the librarian:

- Fees are charged per **day late**: each day after the due date, up to and including the return day.
- The library is closed on **Sundays**, so Sundays are never counted.
- **Books** cost 15p per day, up to a maximum of 500p for one item.
- **DVDs** cost 100p per day, up to a maximum of 1000p.
- **Reference** items can't be borrowed at all, so asking for their fee is an error.
- Members aged **under 16** pay half, rounded down to a whole penny.
- Returned on or before the due date: no fee.

For example, a book due on Friday 6 November 2026 and returned on Tuesday 10 November is late on Saturday, Monday and Tuesday (Sunday doesn't count): 3 days, 45p.

## Forge bonus: testing a ledger

This time you don't write the code: a club treasurer has written `ledger.py`, and your job is to test it thoroughly before the club trusts it with real money.

## Weekly quiz

## Week 24 review

Tick off each one honestly. If one feels shaky, go back and redo two exercises from that day.

- I can explain what tests give you, and write arrange, act, assert tests that catch bugs. ([Day 1](w024d1.html))
- I can run pytest, read its reports, and use `pytest.raises`, `approx` and `parametrize`. ([Day 2](w024d2.html))
- I can choose inputs with classes and boundaries, write good tests, inject dependencies and test properties. ([Day 3](w024d3.html))
- I can work test first: red, green, refactor. ([Day 4](w024d4.html))
- I can work through edge cases and use fixtures, `tmp_path`, `capsys` and `monkeypatch`. ([Day 5](w024d5.html))
- I built a late-fee calculator test first, and tested a ledger I didn't write. (Today)

Tomorrow is a **rest day**. Next week is the **consolidation project**: you'll build an expense tracker from scratch over the week, with its core logic tested, data saved to files, and a command-line interface, using everything from this phase.